In [13]:
"""
==========================================================
Statistical Analysis — Automated vs Ground Truth Cell Counting
HT29 Colon Cancer Cells (BBBC001)
==========================================================
Author  : sai sasi sekhar kongala
Date    : May 2026
Dataset : BBBC001v1 — Broad Bioimage Benchmark Collection
          https://bbbc.broadinstitute.org/BBBC001
 
Tests performed:
  1. Pearson correlation         — strength of linear relationship
  2. R² (coefficient of determination) — variance explained
  3. Paired t-test               — systematic bias between methods
  4. Bland-Altman analysis       — method agreement (gold standard)
  5. Mean absolute error (MAE)   — average counting error in cells
  6. Per-frame % error           — vs official ground truth
  7. Method comparison boxplot   — CellProfiler vs ImageJ vs Python
 
DEPENDENCIES:
    pip install numpy scipy matplotlib pandas
 
USAGE:
    python statistical_analysis.py
==========================================================
"""

'\n==========================================================\nStatistical Analysis — Automated vs Ground Truth Cell Counting\nHT29 Colon Cancer Cells (BBBC001)\n==========================================================\nAuthor  : [Your Name]\nDate    : May 2026\nDataset : BBBC001v1 — Broad Bioimage Benchmark Collection\n          https://bbbc.broadinstitute.org/BBBC001\n\nTests performed:\n  1. Pearson correlation         — strength of linear relationship\n  2. R² (coefficient of determination) — variance explained\n  3. Paired t-test               — systematic bias between methods\n  4. Bland-Altman analysis       — method agreement (gold standard)\n  5. Mean absolute error (MAE)   — average counting error in cells\n  6. Per-frame % error           — vs official ground truth\n  7. Method comparison boxplot   — CellProfiler vs ImageJ vs Python\n\nDEPENDENCIES:\n    pip install numpy scipy matplotlib pandas\n\nUSAGE:\n    python statistical_analysis.py\n===============================

In [27]:
import numpy as np
import matplotlib 
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [28]:
# Data
# Ground truth: two independent human expert counts from BBBC001vt
# Source: https://data.broadinstitute.org/bbbc/BBBC001/BBBC001_v1_counts.txt

frames = ['F0', 'F1', 'F2', 'F3', 'F4', 'F5']
human1        = np.array([350, 336, 396, 320, 398, 241])   # human observer 1
human2        = np.array([362, 342, 447, 341, 533, 257])   # human observer 2
gt            = (human1 + human2) / 2.0                    # ground truth average
 
# My CellProfiler counts (from Image.csv, Count_Nuclei column)
my_counts     = np.array([346, 330, 393, 313, 452, 233])

# Previous methods (for comparison)
imagej_counts = np.array([313, 289, 293, 248, 325, 201])
python_counts = np.array([224, 218, 195, 183, 229, 172])

#per-frame error
errors_pct = np.abs(my_counts - gt)/gt*100

In [22]:
# ── Statistical tests ─────────────────────────────────────────────────────────
 
# 1. Pearson correlation
r, p_pearson = stats.pearsonr(my_counts, gt)
 
# 2. Paired t-test (tests for systematic bias — p>0.05 means no significant bias)
t_stat, p_ttest = stats.ttest_rel(my_counts, gt)
 
# 3. Mean absolute error
mae = np.mean(np.abs(my_counts - gt))
 
# 4. Bland-Altman analysis
means_ba   = (my_counts + gt) / 2
diffs_ba   = my_counts - gt
mean_diff  = np.mean(diffs_ba)
std_diff   = np.std(diffs_ba, ddof=1)
loa_upper  = mean_diff + 1.96 * std_diff
loa_lower  = mean_diff - 1.96 * std_diff
 
# 5. Linear regression
slope, intercept, r_val, p_reg, se = stats.linregress(gt, my_counts)
 
# 6. Human inter-observer variability (for context)
human_mae = np.mean(np.abs(human2 - human1))
 
# ── Print results ─────────────────────────────────────────────────────────────
print("=" * 55)
print("  STATISTICAL RESULTS SUMMARY")
print("=" * 55)
print(f"  Pearson r          : {r:.4f}  (p = {p_pearson:.4f})")
print(f"  R²                 : {r_val**2:.4f}  ({r_val**2*100:.1f}% variance explained)")
print(f"  Paired t-test      : t = {t_stat:.3f}, p = {p_ttest:.4f}")
print(f"  Interpretation     : {'No significant bias (p > 0.05)' if p_ttest > 0.05 else 'Significant systematic bias detected'}")
print(f"  Mean error %       : {np.mean(errors_pct):.2f}%")
print(f"  MAE                : {mae:.2f} cells")
print(f"  Bland-Altman bias  : {mean_diff:.2f} cells (negative = undercounting)")
print(f"  95% Limits of Agr. : [{loa_lower:.1f},  {loa_upper:.1f}]")
print(f"  Human observer MAE : {human_mae:.2f} cells (between expert 1 and 2)")
print("=" * 55)
print(f"\n  Benchmark (Carpenter 2006): 6.2%")
print(f"  My result                 : {np.mean(errors_pct):.1f}%")
print(f"  Human variability ceiling : 11%")
print("=" * 55)

  STATISTICAL RESULTS SUMMARY
  Pearson r          : 0.9957  (p = 0.0000)
  R²                 : 0.9915  (99.1% variance explained)
  Paired t-test      : t = -5.463, p = 0.0028
  Interpretation     : Significant systematic bias detected
  Mean error %       : 4.47%
  MAE                : 15.75 cells
  Bland-Altman bias  : -15.75 cells (negative = undercounting)
  95% Limits of Agr. : [-29.6,  -1.9]
  Human observer MAE : 40.17 cells (between expert 1 and 2)

  Benchmark (Carpenter 2006): 6.2%
  My result                 : 4.5%
  Human variability ceiling : 11%


In [31]:
# ── Figures ───────────────────────────────────────────────────────────────────
colors_frame = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']
 
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'Statistical Analysis — Automated vs Ground Truth Cell Counting\n'
    'HT29 Colon Cancer Cells (BBBC001)',
    fontsize=14, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)
 
# Plot 1 — Count comparison bar chart
ax1 = fig.add_subplot(gs[0, 0])
x, w = np.arange(len(frames)), 0.35
ax1.bar(x - w/2, gt, w, label='Ground truth', color='#4CAF50', alpha=0.85, edgecolor='black', lw=0.5)
ax1.bar(x + w/2, my_counts, w, label='CellProfiler (mine)', color='#2196F3', alpha=0.85, edgecolor='black', lw=0.5)
for i, (g, m) in enumerate(zip(gt, my_counts)):
    ax1.text(i - w/2, g + 6, f'{int(g)}', ha='center', fontsize=8, color='#2e7d32')
    ax1.text(i + w/2, m + 6, f'{m}', ha='center', fontsize=8, color='#1565C0')
ax1.set_xlabel('Frame', fontsize=11); ax1.set_ylabel('Cell count', fontsize=11)
ax1.set_title('Cell count comparison', fontsize=12, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(frames); ax1.legend(fontsize=9); ax1.set_ylim(0, 550)
 
# Plot 2 — Correlation / regression
ax2 = fig.add_subplot(gs[0, 1])
for i, (g, m, fr) in enumerate(zip(gt, my_counts, frames)):
    ax2.scatter(g, m, color=colors_frame[i], s=80, zorder=5, edgecolor='white', lw=0.8)
    ax2.annotate(fr, (g, m), textcoords='offset points', xytext=(6, 4), fontsize=9, color=colors_frame[i])
gt_line = np.linspace(min(gt) - 20, max(gt) + 20, 100)
ax2.plot(gt_line, gt_line, 'k--', lw=1, alpha=0.4, label='Perfect agreement')
ax2.plot(gt_line, slope * gt_line + intercept, color='#E53935', lw=1.5,
         label=f'Regression (R²={r_val**2:.3f})')
ax2.set_xlabel('Ground truth count', fontsize=11)
ax2.set_ylabel('My count (CellProfiler)', fontsize=11)
ax2.set_title('Correlation plot', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.text(0.05, 0.92, f'r = {r:.3f}\np = {p_pearson:.3f}', transform=ax2.transAxes, fontsize=9,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray', alpha=0.8))
 
# Plot 3 — Bland-Altman
ax3 = fig.add_subplot(gs[0, 2])
for i, (mb, db, fr) in enumerate(zip(means_ba, diffs_ba, frames)):
    ax3.scatter(mb, db, color=colors_frame[i], s=80, zorder=5, edgecolor='white', lw=0.8)
    ax3.annotate(fr, (mb, db), textcoords='offset points', xytext=(6, 4), fontsize=9, color=colors_frame[i])
ax3.axhline(mean_diff, color='#E53935', lw=1.5, label=f'Bias = {mean_diff:.1f}')
ax3.axhline(loa_upper, color='#FF7043', lw=1, ls='--', label=f'+1.96 SD = {loa_upper:.1f}')
ax3.axhline(loa_lower, color='#FF7043', lw=1, ls='--', label=f'-1.96 SD = {loa_lower:.1f}')
ax3.axhline(0, color='black', lw=0.5, alpha=0.3)
ax3.fill_between([min(means_ba) - 20, max(means_ba) + 20], loa_lower, loa_upper, alpha=0.07, color='#FF7043')
ax3.set_xlabel('Mean of GT & my count', fontsize=11)
ax3.set_ylabel('Difference (mine − GT)', fontsize=11)
ax3.set_title('Bland–Altman plot', fontsize=12, fontweight='bold')
ax3.legend(fontsize=8, loc='upper right')
 
# Plot 4 — Per-frame error
ax4 = fig.add_subplot(gs[1, 0])
bar_colors = ['#4CAF50' if e < 5 else '#FF9800' for e in errors_pct]
bars = ax4.bar(frames, errors_pct, color=bar_colors, edgecolor='black', lw=0.5, alpha=0.85)
ax4.axhline(np.mean(errors_pct), color='#E53935', ls='--', lw=1.5, label=f'Mean = {np.mean(errors_pct):.1f}%')
ax4.axhline(11, color='gray', ls=':', lw=1, label='Human variability (11%)')
ax4.axhline(6.2, color='#9C27B0', ls=':', lw=1, label='Benchmark (6.2%)')
for bar, e in zip(bars, errors_pct):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f'{e:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax4.set_xlabel('Frame', fontsize=11); ax4.set_ylabel('Error %', fontsize=11)
ax4.set_title('Per-frame error vs benchmarks', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8); ax4.set_ylim(0, 16)
 
# Plot 5 — Method comparison boxplot
ax5 = fig.add_subplot(gs[1, 1])
data_box   = [gt, my_counts, imagej_counts, python_counts]
labels_box = ['Ground\ntruth', 'CellProfiler\n(mine)', 'ImageJ\n(first)', 'Python\n(first)']
colors_box = ['#4CAF50', '#2196F3', '#FF9800', '#9E9E9E']
bp = ax5.boxplot(data_box, patch_artist=True, notch=False, widths=0.5)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for median in bp['medians']:
    median.set_color('black'); median.set_linewidth(2)
ax5.set_xticklabels(labels_box, fontsize=9)
ax5.set_ylabel('Cell count', fontsize=11)
ax5.set_title('Method comparison (all frames)', fontsize=12, fontweight='bold')
for i, d in enumerate(data_box):
    ax5.text(i + 1, np.median(d) - 12, f'Med={int(np.median(d))}', ha='center', fontsize=8)
 
# Plot 6 — Summary table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
table_data = [
    ['Metric', 'Value', 'Interpretation'],
    ['Pearson r',       f'{r:.3f}',              'Near-perfect correlation'],
    ['R²',              f'{r_val**2:.3f}',        f'{r_val**2*100:.1f}% variance explained'],
    ['Paired t p',      f'{p_ttest:.3f}',         'p<0.05 = systematic bias'],
    ['Mean error %',    f'{np.mean(errors_pct):.1f}%', 'Within human variability'],
    ['Bias (B-A)',       f'{mean_diff:.1f} cells', 'Slight undercounting'],
    ['95% LoA',         f'[{loa_lower:.0f}, {loa_upper:.0f}]', 'Limits of agreement'],
    ['MAE',             f'{mae:.1f} cells',       'Mean absolute error'],
    ['Benchmark',       '6.2%',                   'Carpenter et al. 2006'],
    ['My result',       '7.1%',                   'Within human variability (11%)'],
]
tbl = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center',
                colWidths=[0.38, 0.25, 0.37])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 1.5)
for (r_idx, c_idx), cell in tbl.get_celld().items():
    if r_idx == 0:
        cell.set_facecolor('#1565C0'); cell.set_text_props(color='white', fontweight='bold')
    elif r_idx % 2 == 0:
        cell.set_facecolor('#EEF2FF')
    else:
        cell.set_facecolor('#F8F9FF')
    cell.set_edgecolor('#CCCCCC')
ax6.set_title('Statistical summary table', fontsize=12, fontweight='bold', pad=10)
os.makedirs('results', exist_ok=True)

plt.savefig('results/statistical_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
print("\n[SAVED] results/statistical_analysis.png")
 


[SAVED] results/statistical_analysis.png
